# HapMap3 panel for the round-1 set (ancestry PCA input)

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run, repo_root
from aoucov.batch import run_script
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

In [ ]:
from stage_helpers import verify_panel
STAGE = "hm3_panel"
require_ready(cfg, STAGE)
rep, OUT = Report(cfg, STAGE), single_target(cfg, STAGE)
PREFIX = OUT[:-4]
geno_key = "acaf_bfile" if "acaf_bfile" in cfg.params["inputs"] else "acaf_pfile"
panel_env = dict(PLINK2=cfg.tool("plink2"), PLINK=cfg.tool("plink"),
                 GENO_FLAG="--bfile" if geno_key == "acaf_bfile" else "--pfile",
                 GENO_PATTERN=cfg.input(geno_key, chr="{chr}"), NCHR=22,
                 EXTRACT=cfg.input("hm3_ids"), MAF=cfg.param("qc_maf"), HWE=cfg.param("qc_hwe"),
                 GENO=cfg.param("qc_geno"), THREADS=THREADS)

In [ ]:
if needs_run(OUT, FORCE):
    with record(cfg, STAGE, OUT, tools={"plink2": cfg.tool("plink2")}):
        run_script(cfg, "panel.sh", {**panel_env, "KEEP": single_target(cfg, "ancestry_round1"),
                                     "REF_BFILE": cfg.input("ref_bfile"), "OUT": PREFIX})

## Checks

In [ ]:
counts = pd.read_csv(PREFIX + ".counts.tsv", sep="\t")
rep.count("n_samples", counts.set_index("step").loc["samples", "variants"])
steps = counts[counts["step"] != "samples"]
rep.table(steps, "variants_by_step")
rep.note(n_variants=int(steps["variants"].iloc[-1]))
fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(steps["step"], steps["variants"]); ax.invert_yaxis(); ax.set_xlabel("variants")
rep.fig(fig, "variants_by_step"); plt.show()

In [ ]:
verify_panel(rep, PREFIX, cfg.tool("plink2"), cfg.param("qc_maf"), cfg.param("qc_geno"),
             cfg.param("qc_hwe"), "the round-1 set")
rep.check("matched to the reference on ID+REF+ALT", "matched_to_reference" in set(steps["step"]))
rep.save()